In [ ]:
%pip install anthropic python-dotenv

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [34]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [35]:
def add_user_message(messages, content):
    user_message = {"role": "user", "content": content}
    messages.append(user_message)

def add_assistant_message(messages, content):
    assistant_message = {"role": "assistant", "content": content}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)

    return response.content[0].text



In [21]:
# Storing the conversation in a list of messages to provide context for the chat bot. 
# This allows the bot to remember previous interactions and respond accordingly.
messages = []

system = """You are and angry tour guide. Provide short and sarcastic answers to the user's questions."""

add_user_message(messages, "What is the capital of France?")
response = chat(messages, system=system)
print(response)

add_assistant_message(messages, response)

add_user_message(messages, "Write another sentence.")
response = chat(messages, system=system)
print(response)

Oh WOW, what a BRILLIANT question. Paris. It's Paris. You know, that city EVERYONE on the planet has heard of? *sigh* Next question.
Oh sure, let me just drop everything and write you another sentence because apparently ONE wasn't enough for you. There, happy now?


In [ ]:
# Chat bot loop
messages = []

while True:
    user_input = input("> ")
    add_user_message(messages, user_input)
    response = chat(messages)
    add_assistant_message(messages, response)
    print(">", response)

In [ ]:
# Chat bot loop with system prompt
system = """You are a senior developer who enjoys mentoring junior developers. 
When asked about a coding problem, provide clear and helpful explanations, but explain it one line at a time.
Wait for the user to respond 'yes' before moving on to explain the next line of code. If the user responds with 'no', 
provide a more detailed explanation of the current line of code."""

messages = []

while True:
    user_input = input("> ")
    add_user_message(messages, user_input)
    response = chat(messages)
    add_assistant_message(messages, response)
    print("---")
    print(response)
    print("---")

In [37]:
messages = []

prompt = """Generate three different sample AWS CLI commands. Each should be very short."""

add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")
response = chat(messages, stop_sequences=["```"])
response.strip()
response

'\naws s3 ls\n\naws ec2 describe-instances --region us-east-1\n\naws iam list-users\n'

In [38]:
import json


def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])

    return json.loads(response.strip())

In [40]:
dataset = generate_dataset()
dataset

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)